# Разведочный анализ через pygwalker

Визуальная разведка перед тем как фиксировать гипотезы для двух основных исследований (ABC/XYZ по товарам, RFM и когорты по клиентам). Оба исследования строятся на данных за 2023 год, поэтому выборки ниже тоже ограничены 2023-м чтобы разведка соответствовала основным исследованиям. Один блок (про пул `client_id`) намеренно на полной истории там смысл виден только на нескольких годах.

Виджеты `pyg.walk()` ниже рендерятся только в живой Jupyter-сессии. На GitHub в этих ячейках пусто, смотреть нужно локально: `jupyter lab analysis/00_pygwalker_eda.ipynb`.

In [ ]:
import sys
for p in (".", ".."):
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
from sqlalchemy import create_engine

from scripts.common import config

engine = create_engine(
    f"postgresql+psycopg2://{config.POSTGRES_USER}:{config.POSTGRES_PASSWORD}"
    f"@{config.POSTGRES_HOST}:{config.POSTGRES_PORT}/{config.POSTGRES_DB}"
)

YEAR_START = "2023-01-01"
YEAR_END = "2024-01-01"

## Транзакции (выборка)

30 случайных дат 2023 года из `staging.stg_sales`.

In [ ]:
df_txn = pd.read_sql(
    f"""
    select *
    from staging.stg_sales
    where source_date in (
        select source_date
        from (
            select distinct source_date from staging.stg_sales
            where source_date >= '{YEAR_START}' and source_date < '{YEAR_END}'
        ) t
        order by random()
        limit 30
    )
    """,
    engine,
)
df_txn.shape

In [ ]:
import pygwalker as pyg

pyg.walk(df_txn)

## Товары

Выручка количество и число продаж по товару за 2023 год, посчитано из `staging.stg_sales`.

In [ ]:
df_products = pd.read_sql(
    f"""
    select
        product_id,
        sum(total_price) as total_revenue,
        sum(quantity) as total_quantity,
        count(*) as sales_count,
        avg(price_per_item) as avg_price_per_item
    from staging.stg_sales
    where source_date >= '{YEAR_START}' and source_date < '{YEAR_END}'
    group by product_id
    """,
    engine,
)
df_products.shape

In [ ]:
pyg.walk(df_products)

## Клиенты (выборка)

50 тысяч случайных клиентов активных в 2023 году.

In [ ]:
df_customers = pd.read_sql(
    f"""
    select
        client_id,
        sum(total_price) as total_revenue,
        sum(quantity) as total_quantity,
        count(*) as sales_count,
        count(distinct product_id) as unique_products,
        min(source_date) as first_purchase_date,
        max(source_date) as last_purchase_date
    from staging.stg_sales
    where source_date >= '{YEAR_START}' and source_date < '{YEAR_END}'
    group by client_id
    order by random()
    limit 50000
    """,
    engine,
)
df_customers.shape

In [ ]:
pyg.walk(df_customers)

## Продажи по дням

`marts.mart_daily_sales` за 2023 год эта витрина посуточная, фильтр по дате навешан сверху.

In [ ]:
df_daily = pd.read_sql(
    f"""
    select * from marts.mart_daily_sales
    where source_date >= '{YEAR_START}' and source_date < '{YEAR_END}'
    order by source_date
    """,
    engine,
)
df_daily.shape

In [ ]:
pyg.walk(df_daily)

## Ставка скидки на позицию

`discount_per_item / price_per_item` на выборке транзакций 2023 В целом по данным (не только 2023) это распределение равномерное от 0 до 1, а не сосредоточено вокруг типичных для ритейла 10-30%.

In [ ]:
import matplotlib.pyplot as plt

discount_rate = df_txn["discount_per_item"] / df_txn["price_per_item"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(discount_rate, bins=50)
ax.set_xlabel("discount_per_item / price_per_item, выборка 2023")
ax.set_ylabel("количество позиций")
plt.show()

## Пул client_id (полная история, не только 2023)

Единственный блок который специально не ограничен 2023 годом: явление видно только на нескольких годах сразу для когортного анализа по 2023 году знать про него всё равно нужно.

In [ ]:
new_customers_by_year = pd.read_sql(
    """
    select extract(year from first_purchase_date) as yr, count(*) as new_customers
    from marts.mart_customer_activity
    group by 1
    order by 1
    """,
    engine,
)
new_customers_by_year

За 2022 год впервые появилось большинство `client_id` которые вообще когда-либо встретятся в данных к 2023-2024 пул уже почти исчерпан 
Для когортного анализа по 2023 году это значит, что часть клиентов в когортах 2023 года на самом деле не новые, они впервые купили ещё в 2022-м, а в 2023-й год просто первый раз попадают в рамки самого исследования. Разница между новым клиентом и впервые попавшим в окно в `02_customers_ltv.ipynb` должна быть явной.

## Что учитывать дальше

Скидка на позицию распределена равномерно от 0 до 100% от цены независимо от товара. Это значит что реальной зависимости скорее всего не найдётся: скидка не сигнал, а шум.

Выручка по товарам распределена ближе к колоколообразной форме чем к длинному хвосту. Парето-скос 80/20 в ABC-анализе под вопросом.

Пул `client_id` почти исчерпан уже к 2023 году. Для RFM и когорт по 2023 году это значит, что новый клиент в 2023 и первая покупка в данных вообще - разные вещи, стоит явно указать.

`gender` не привязан к клиенту: у одного и того же `client_id` в разных транзакциях встречаются оба значения. Для RFM-сегментации это поле не годится как признак клиента.

`purchase_date` из payload API не всегда совпадает с `source_date` (расхождение иногда в месяцы). Для RFM и когорт даты нужно считать по `source_date`, не по `purchase_date`.